# Train captcha2-resolver on Colab

`Runtime → Change runtime type → T4 GPU`. This notebook only calls the `c2r` CLI; all logic lives in the package.

Needs approved access to the gated dataset and an `HF_TOKEN` in Colab secrets (🔑 sidebar).

In [ ]:
REPO_URL = "https://github.com/vinc3r3/captcha2-resolver.git"
HF_DATASET_REPO = "<you>/captcha2-dataset"

!git clone --depth 1 $REPO_URL /content/captcha2-resolver
%cd /content/captcha2-resolver

## Install
Paddle GPU build (bundles CUDA 12.6). Colab's preinstalled torch breaks once Paddle swaps in its own NCCL (`undefined symbol ncclCommWindow*`), and paddleocr imports torch if present, so remove it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip uninstall -y -q torch torchvision torchaudio
%pip install -q -e ".[train,export,data]"

## Data

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_DATASET_REPO"] = HF_DATASET_REPO
!c2r hub pull-dataset
!c2r data merge
!c2r data validate | tail -12

## Train → export → evaluate

In [ ]:
!c2r train prepare --epochs 100 --num-workers 2

In [ ]:
!c2r train run      # add --resume to continue from work/output/latest

In [ ]:
!c2r export onnx --check data/images/$(ls data/images | head -1)
!c2r evaluate

## Download

In [ ]:
import shutil

from google.colab import files

files.download(shutil.make_archive("/content/captcha_onnx", "zip", "models/onnx"))